SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.

SPDX-License-Identifier: Apache-2.0

# Lab 1: Vision Language Models

## Cosmos 3 Reasoner

In this notebook, we'll explore Vision Language Models (VLMs) hands-on using **Cosmos 3**, NVIDIA's reasoning VLM for physical AI. We'll deploy the model as a NIM microservice, run inference on images and videos, learn how to prompt it effectively, and measure its performance - everything you need before moving on to dataset generation and fine-tuning in the next notebooks.

### Learning Objectives:
This notebook explores the following topics:
* What VLMs are and the Cosmos 3 model family
* How to deploy the Cosmos 3 Reasoner NIM
* Single image and video inference through the OpenAI library
* How vision token count affects output quality and latency
* Good vs bad prompts for VLMs
* Structured outputs: formatted reports, JSON, true/false and multiple choice
* Reasoning vs non-reasoning mode
* Benchmarking: latency, throughput, and the knobs that drive them
* How to tear down the deployment when finished

### Table of Contents

**[Set Up the Environment](#Set-Up-the-Environment)**  
**[Part 1: Understanding VLMs and Cosmos 3](#Part-1:-Understanding-VLMs-and-Cosmos-3)**  
&nbsp;&nbsp;&nbsp;&nbsp;[1.1 What is a VLM?](#1.1-What-is-a-VLM?)  
&nbsp;&nbsp;&nbsp;&nbsp;[1.2 The Cosmos 3 Family](#1.2-The-Cosmos-3-Family)  
**[Part 2: Deploy Cosmos 3 Reasoner](#Part-2:-Deploy-Cosmos-3-Reasoner)**  
**[Part 3: Image Inference](#Part-3:-Image-Inference)**  
**[Part 4: Video Inference](#Part-4:-Video-Inference)**  
**[Part 5: Vision Tokens: Impact on Quality & Latency](#Part-5:-Vision-Tokens:-Impact-on-Quality-&-Latency)**  
**[Part 6: Good vs Bad Prompts](#Part-6:-Good-vs-Bad-Prompts)**  
**[Part 7: Structured Outputs & Tasks](#Part-7:-Structured-Outputs-&-Tasks)**  
&nbsp;&nbsp;&nbsp;&nbsp;[7.1 Formatted Reports](#7.1-Formatted-Reports)  
&nbsp;&nbsp;&nbsp;&nbsp;[7.2 JSON Events](#7.2-JSON-Events)  
&nbsp;&nbsp;&nbsp;&nbsp;[7.3 True/False and Multiple Choice](#7.3-True/False-and-Multiple-Choice)  
**[Part 8: Reasoning vs Non-Reasoning](#Part-8:-Reasoning-vs-Non-Reasoning)**  
**[Part 9: Benchmarking Performance](#Part-9:-Benchmarking-Performance)**  
&nbsp;&nbsp;&nbsp;&nbsp;[9.1 Latency: Time to First Token and Output Length](#9.1-Latency:-Time-to-First-Token-and-Output-Length)  
&nbsp;&nbsp;&nbsp;&nbsp;[9.2 Input Size: Sampled Frames](#9.2-Input-Size:-Sampled-Frames)  
&nbsp;&nbsp;&nbsp;&nbsp;[9.3 Concurrency and Throughput](#9.3-Concurrency-and-Throughput)  
**[Part 10: Tear Down](#Part-10:-Tear-Down)**  
**[Review](#Review)**

---
### Set Up the Environment

Cosmos 3 is deployed as a **NIM microservice** and exposes an **OpenAI-compatible API**. This means we can interact with it using the standard `openai` Python library.

Let's start by installing dependencies and defining our endpoint and test media.

In [ ]:
host_ip = "0.0.0.0"

In [ ]:
# Endpoint for our deployed model
vlm_base_url = f"http://{host_ip}:30082"  # Cosmos 3 Reasoner NIM endpoint

# Sample media for testing
test_image = "data/traffic.png"
test_video = "data/traffic_short.mp4"

In [ ]:
import sys
python_exe = sys.executable
!{python_exe} -m ensurepip --upgrade
!{python_exe} -m pip install -r requirements.txt 

In [ ]:
import json
import time
import base64
import requests
from openai import OpenAI
from IPython.display import Markdown, display, Image, Video

The following helper function will verify API responses and help debug if any errors occur.

In [ ]:
#helper function to verify responses
def check_response(response, text=False):
    print(f"Response Code: {response.status_code}")
    if response.status_code == 200:
        print("Response Status: Success")
        if text:
            print(response.text)
            return response.text
        else:
            print(json.dumps(response.json(), indent=4))
            return response.json()
    else:
        print("Response Status: Error")
        print(response.text)
        return None 

---
### Part 1: Understanding VLMs and Cosmos 3

#### 1.1 What is a VLM?

A **Vision Language Model (VLM)** is an AI model that accepts both visual inputs (images, video frames) and text. You provide it with an image or video clip along with a text prompt, and it responds with text. Internally, a vision encoder converts pixels into tokens that the language model processes alongside your prompt. The important distinction: VLMs can interpret visual content, which a standard text-only Large Language Model (LLM) cannot do.

**Cosmos 3** is NVIDIA's open omnimodal world model family for physical AI and robotics. The capabilities most relevant to this lab are:

- **Spatial-temporal understanding**: it reasons about space, time, and physics across video frames, not just individual images
- **Chain-of-thought reasoning**: using `<think>` tags, the model works through complex scenes step by step before producing a final answer
- **Dense captioning** with **temporal localization**: it produces detailed descriptions of video scenes and can output timestamped events in JSON format
- Cosmos 3 also supports 2D grounding (bounding boxes) and embodied reasoning (next-action prediction), though those capabilities are outside the scope of this lab

#### 1.2 The Cosmos 3 Family

The full Cosmos 3 model actually contains **two towers** in every transformer layer: an autoregressive **Reasoner** tower (the VLM) and a diffusion **Generator** tower (for video generation). The Reasoner tower can run standalone (the Generator always activates both towers), and what we deploy in this lab is the **Reasoner tower only** - this is what the "Reasoner" NIM serves. You will meet the Generator side of Cosmos in the next notebook, where we use it to generate and augment video datasets.

Cosmos 3 comes in three sizes:

| Model | Total Params | Reasoner Tower | Base Architecture | Target Platform |
|-------|-------------|----------------|-------------------|-----------------|
| Cosmos3-Edge | 4B | 2B | From scratch (Nemotron-style) | Jetson / on-device |
| Cosmos3-Nano | 16B | 8B | Qwen3-VL-8B | Workstation |
| Cosmos3-Super | 64B | 32B | Qwen3-VL-32B | Datacenter |

This is why you will see the same model referred to as both "Nano (16B)" and "Nano Reasoner (8B)" - the NIM checkpoint contains just the 8B Reasoner tower of the 16B model. In this lab, we use **Nano**.

---
### Part 2: Deploy Cosmos 3 Reasoner

The Cosmos 3 Reasoner NIM serves both the Nano and Super variants from a single container image (`nvcr.io/nim/nvidia/cosmos3-reasoner`); the size is selected at startup with the `NIM_MODEL_SIZE` environment variable.

Deploying a NIM takes one `docker run` command. For reference, this is the full deployment:

```bash
docker run -d --name cosmos3-reasoner \
    --gpus all --shm-size=16GB \
    -e NGC_API_KEY \
    -e NIM_MODEL_SIZE=nano \
    -p 30082:8000 \
    -v ~/.cache/nim:/opt/nim/.cache \
    nvcr.io/nim/nvidia/cosmos3-reasoner:1.7
```

The `NGC_API_KEY` is only needed to download the container and model weights; once the NIM is running, its API requires no authentication.

**In this lab environment, the service has already been deployed for you** - let's verify it is running.

Let's verify the model service is running. **Make sure the cell outputs "Response Code: 200" before proceeding.**

<div class="alert alert-block alert-info">
        <b>Note:</b> The model service may take several minutes to start up as it downloads and initializes model weights.
    </div>

In [ ]:
# Check VLM (Cosmos 3) health
try:
    resp = requests.get(vlm_base_url + "/v1/health/ready")
    resp = check_response(resp)
except Exception as e:
    print(f'VLM Server not ready: {e}')

Now let's create an OpenAI client for the model and retrieve the configured model name.

In [ ]:
# Create OpenAI-compatible client
vlm_client = OpenAI(base_url=vlm_base_url + "/v1", api_key="EMPTY")

# Retrieve available models
try:
    vlm_model = vlm_client.models.list().data[0].id
    print(f"VLM Model: {vlm_model}")
except Exception:
    raise RuntimeError("Could not reach the VLM endpoint - make sure the health check above returned 200 before continuing.") from None

---
### Part 3: Image Inference

The simplest way to use Cosmos 3 is to pass a single image and ask a question about it. The image is sent as a base64-encoded string or a file URL in the message content.

Let's start with a basic image description.

In [ ]:
# Display the test image
Image(filename=test_image, width=800)

In [ ]:
# Helper: encode image to base64 for API calls
def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

To send an image to the VLM, we use the OpenAI chat completions API with `image_url` content type. The API expects the image as a base64-encoded data URL or a file path.

<div class="alert alert-block alert-info">
        <b>Important:</b> In the message content, media (images/videos) should be listed <b>before</b> text to match the model's training format.
    </div>

In [ ]:
image_b64 = encode_image(test_image)

start_t = time.time()
response = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
                {"type": "text", "text": "Describe this image in detail. What objects, people, and activities do you see?"},
            ],
        },
    ],
    max_tokens=1024,
    temperature=0.4,
    top_p=0.4,
)
response_time = time.time() - start_t

image_caption = response.choices[0].message.content
print(f"Response Time: {response_time:.2f} seconds\n")
print(image_caption)

We can also ask specific questions about the image. The VLM will ground its answers in what it actually sees.

In [ ]:
response = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Answer accurately based on the image."},
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
                {"type": "text", "text": "What type of environment is this? Is it safe to proceed forward?"},
            ],
        },
    ],
    max_tokens=512,
    temperature=0.4,
    top_p=0.4,
)

print(response.choices[0].message.content)

---
### Part 4: Video Inference

Cosmos 3 can also process video. When a video is provided, the model samples frames from it and processes them together, giving the model temporal understanding of what's happening over time.

<div class="alert alert-block alert-info">
        <b>Note:</b> By default, the NIM samples video at 4 FPS. Frame sampling can be adjusted per request through the <code>media_io_kwargs</code> parameter - we will use this in the benchmarking section.
    </div>

In [ ]:
#Video(test_video, width=800, embed=True)

For video inference, the `video_url` content type is used. The video can be provided as a file path.

In [ ]:
import os
import time
import base64

# Absolute path to your local video
video_path = os.path.abspath(test_video)

# --- Encode video file as base64 ---
with open(video_path, "rb") as f:
    video_bytes = f.read()

video_b64 = base64.b64encode(video_bytes).decode("utf-8")

# Create a proper data URL (adjust mime type if needed)
video_data_url = f"data:video/mp4;base64,{video_b64}"

# --- Send request ---
start_t = time.time()

response = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {
                    "type": "video_url",
                    "video_url": {
                        "url": video_data_url
                    },
                },
                {
                    "type": "text",
                    "text": "Caption the video in detail. Describe the scene, objects, and any events.",
                },
            ],
        },
    ],
    max_tokens=1024,
    temperature=0.4,
    top_p=0.4,
)

response_time = time.time() - start_t

video_caption = response.choices[0].message.content

print(f"Response Time: {response_time:.2f} seconds\n")
print(video_caption)

---
### Part 5: Vision Tokens: Impact on Quality & Latency

When Cosmos 3 processes an image or video, it converts the visual input into **vision tokens**. The number of vision tokens directly impacts:

1. **Quality** - More tokens = higher resolution = finer details
2. **Latency** - More tokens = more processing time
3. **Memory** - More tokens = more GPU memory required

The vision token count is controlled by the image/video resolution and the number of frames:

```
Vision Tokens ≈ (Height × Width) / (32 × 32) × Frames / 2
```

The `/ 2` comes from temporal compression: Cosmos 3 merges pairs of adjacent frames into one set of tokens. For a single image, the frames and temporal terms drop out and the count is simply `(Height × Width) / (32 × 32)`.

For reference, here are some common configurations:

| Height | Width | Frames | Vision Tokens | Recommended Use |
|--------|-------|--------|---------------|----------------|
| 448    | 448   | 80     | ~7,840        | Production default shape |
| 540    | 960   | 30     | ~7,594        | Balanced |
| 720    | 1280  | 20     | ~9,000        | High quality |
| 1080   | 1920  | 8      | ~8,100        | Ultra-high res, fewer frames |

<div class="alert alert-block alert-info">
        <b>Recommendation:</b> Keep vision tokens below 16,000. Going beyond will significantly increase processing time and memory usage.
    </div>

Let's demonstrate this by running the same image at different token budgets and comparing the results.

In [ ]:
# LOW vision tokens - fast but fewer details
start_t = time.time()
response_low = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{image_b64}"},
                    "min_pixels": 256 * 1024,   # 256 tokens worth of pixels
                    "max_pixels": 512 * 1024,    # Cap at 512 tokens
                },
                {"type": "text", "text": "Describe this image in detail. List every object and person you can identify."},
            ],
        },
    ],
    max_tokens=1024,
    temperature=0.4,
    top_p=0.4,
)
low_time = time.time() - start_t
low_output = response_low.choices[0].message.content

print(f"LOW Token Budget - Response Time: {low_time:.2f}s")
print(f"Output:\n{low_output}")

In [ ]:
# HIGH vision tokens - slower but more details
start_t = time.time()
response_high = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{image_b64}"},
                    "min_pixels": 4096 * 1024,   # 4096 tokens worth of pixels
                    "max_pixels": 8192 * 1024,    # Cap at 8192 tokens
                },
                {"type": "text", "text": "Describe this image in detail. List every object and person you can identify."},
            ],
        },
    ],
    max_tokens=1024,
    temperature=0.4,
    top_p=0.4,
)
high_time = time.time() - start_t
high_output = response_high.choices[0].message.content

print(f"HIGH Token Budget - Response Time: {high_time:.2f}s")
print(f"Output:\n{high_output}")

In [ ]:
markdown_string = f"""
<div style="display: flex; gap: 20px;">
  <div style="flex: 1;">
  <h2> Low Vision Tokens </h2>
  <h3> Response Time: {low_time:.2f}s </h3>
    {low_output}
  </div>
  <div style="flex: 1;">
  <h2> High Vision Tokens </h2>
  <h3> Response Time: {high_time:.2f}s </h3>
    {high_output}
  </div>
</div>
"""

Markdown(markdown_string)

Compare the two outputs. The high token budget likely caught more objects and finer details, while the low budget was faster but may have missed things.

This is the fundamental trade-off of every production VLM pipeline: more pixels and more frames produce better results, but at the cost of increased latency and memory. We will measure this directly in the benchmarking section.

---
### Part 6: Good vs Bad Prompts

VLMs are only as good as the prompts you give them. A vague prompt produces a vague answer; a specific prompt that tells the model **what role to take**, **what to focus on**, and **what format to answer in** produces dramatically better results.

Let's run the same video through a bad prompt and a good prompt and compare.

In [ ]:
# BAD prompt - vague, no role, no focus, no format
start_t = time.time()
response_bad = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": video_data_url}},
                {"type": "text", "text": "What's in this video?"},
            ],
        },
    ],
    max_tokens=1024,
    temperature=0.4,
    top_p=0.4,
)
bad_time = time.time() - start_t
bad_output = response_bad.choices[0].message.content

print(f"Response Time: {bad_time:.2f}s\n")
print(bad_output)

In [ ]:
# GOOD prompt - role, focus areas, and output format
good_prompt = """You are a traffic operations monitor reviewing intersection camera footage.

Watch the video and report:
1. Every vehicle visible, its type (car, truck, bus, motorcycle), and what it is doing
2. Any pedestrians or cyclists, and whether they are crossing safely
3. Any traffic incidents or hazards (stalled vehicles, collisions, near-misses, red-light running, jaywalking)

Be specific about where in the frame each observation occurs. If something is not visible, say so - do not guess."""

start_t = time.time()
response_good = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a traffic operations monitor."},
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": video_data_url}},
                {"type": "text", "text": good_prompt},
            ],
        },
    ],
    max_tokens=1024,
    temperature=0.4,
    top_p=0.4,
)
good_time = time.time() - start_t
good_output = response_good.choices[0].message.content

print(f"Response Time: {good_time:.2f}s\n")
print(good_output)

Compare the two outputs. The bad prompt gets you a generic description; the good prompt gets you an inspection report.

The pattern that works for VLM prompting:
- **Role**: tell the model who it is ("You are a traffic operations monitor")
- **Focus**: list exactly what to look for - the model reports what you ask about and skips what you don't
- **Format**: specify how the answer should be structured
- **Honesty**: instruct the model to say when something is not visible, which reduces hallucination

Keep this in mind for the next notebook as well - the same principles apply when writing prompts for video generation.

---
### Part 7: Structured Outputs & Tasks

Free-text answers are fine for humans, but production systems need output they can parse. Cosmos 3 follows formatting instructions well, which lets us turn it into a component that emits **reports**, **JSON**, or **single-token verdicts** on demand.

#### 7.1 Formatted Reports

Let's ask for a structured markdown report instead of a free-form caption. Note how the prompt specifies the exact sections the report should contain.

In [ ]:
report_prompt = """Watch the video and produce a traffic report in Markdown with exactly these sections:

## Summary
One paragraph describing the intersection and overall traffic activity.

## Timeline
A bulleted list of notable events in chronological order.

## Incidents & Hazards
A bulleted list of any traffic incidents or safety concerns. If there are none, state that explicitly."""

start_t = time.time()
response = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a video analytics report generator."},
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": video_data_url}},
                {"type": "text", "text": report_prompt},
            ],
        },
    ],
    max_tokens=1024,
    temperature=0.4,
    top_p=0.4,
)
report_time = time.time() - start_t

print(f"Response Time: {report_time:.2f}s\n")
display(Markdown(response.choices[0].message.content))

---
#### 7.2 JSON Events

Cosmos 3 can produce **temporally localized** captions - describing events with timestamps in structured JSON format. This is the type of output that video analytics pipelines store and index for search.

In [ ]:
response = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {
                    "type": "video_url",
                    "video_url": {
                        "url": video_data_url
                    },
                },
                {"type": "text", "text": "Describe the notable events in the provided video. Provide the result in json format with 'mm:ss.ff' format for time depiction for each event. Use keywords 'start', 'end' and 'caption' in the json output."},
            ],
        },
    ],
    max_tokens=2048,
    temperature=0.4,
    top_p=0.4,
)

print(response.choices[0].message.content)

---
#### 7.3 True/False and Multiple Choice

The cheapest possible VLM output is a **single token**. True/false questions power real-time alerting (is the condition met, yes or no?), and multiple choice is the standard format for VLM evaluation benchmarks - you will meet it again in the fine-tuning notebook when we evaluate models.

Because the output is one token, these calls are also dramatically faster than captioning - compare the response times.

In [ ]:
# True/False - the alerting pattern
tf_question = "Does the video show any collision, near-miss, or stalled vehicle? Answer with only 'yes' or 'no'."

start_t = time.time()
response = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a traffic monitoring system. Answer only with 'yes' or 'no'."},
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": video_data_url}},
                {"type": "text", "text": tf_question},
            ],
        },
    ],
    max_tokens=10,
    temperature=0.2,
    top_p=0.2,
)
tf_time = time.time() - start_t

print(f"Response Time: {tf_time:.2f}s\n")
print(f"Question: {tf_question}")
print(f"Verdict: {response.choices[0].message.content}")

In [ ]:
# Multiple choice - the evaluation pattern
mcq_question = """How many vehicles are visible in the video?

A) None
B) 1-3
C) 4-10
D) More than 10

Answer with only the letter of the correct option."""

start_t = time.time()
response = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are answering a multiple choice question. Respond with only the option letter."},
        {
            "role": "user",
            "content": [
                {"type": "video_url", "video_url": {"url": video_data_url}},
                {"type": "text", "text": mcq_question},
            ],
        },
    ],
    max_tokens=5,
    temperature=0.2,
    top_p=0.2,
)
mcq_time = time.time() - start_t

print(f"Response Time: {mcq_time:.2f}s\n")
print(f"Question: {mcq_question}\n")
print(f"Answer: {response.choices[0].message.content}")

---
### Part 8: Reasoning vs Non-Reasoning

One of Cosmos 3's most powerful features is **chain-of-thought reasoning**. When enabled, the model first thinks through the problem step-by-step inside `<think>` tags before writing its final answer immediately after the `</think>` tag.

This is especially useful for:
- Complex scene understanding
- Safety analysis (incident detection, hazard identification)
- Causal reasoning (what happened and why)

Reasoning is prompt-driven: to enable it, we append the reasoning prompt template to the user message. Let's ask the same question **with and without** reasoning and compare both the answers and the cost.

In [ ]:
# The Cosmos 3 reasoning prompt template
REASONING_PROMPT = """Answer the question using the following format:

<think>
Your reasoning.
</think>

Write your final answer immediately after the </think> tag."""

print(REASONING_PROMPT)

In [ ]:
question = "What can be the next immediate action?"

# WITHOUT reasoning
start_t = time.time()
response = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
                {"type": "text", "text": question},
            ],
        },
    ],
    max_tokens=512,
    temperature=0.4,
    top_p=0.4,
)
plain_time = time.time() - start_t
plain_output = response.choices[0].message.content

print(f"Response Time: {plain_time:.2f} seconds\n")
print(plain_output)

In [ ]:
# WITH reasoning
start_t = time.time()
response = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
                {"type": "text", "text": f"{question}\n\n{REASONING_PROMPT}"},
            ],
        },
    ],
    max_tokens=4096,
    temperature=0.6,
    top_p=0.95,
)
reasoning_time = time.time() - start_t
reasoning_output = response.choices[0].message.content

print(f"Response Time: {reasoning_time:.2f} seconds\n")
print(reasoning_output)

Compare the two responses. Without reasoning, the model answers directly. With reasoning, it first works through its analysis inside the `<think>` tags before producing a concise final answer after the `</think>` tag - typically more grounded and more accurate on complex scenes, but at the cost of generating many more tokens (compare the response times).

That trade-off is the rule of thumb: **use reasoning when accuracy on complex judgments matters, skip it when speed matters** (like the single-token verdicts from Part 7).

<div class="alert alert-block alert-info">
        <b>Tip:</b> When enabling reasoning, increase <code>max_tokens</code> to at least 1024 to allow room for the reasoning trace. Also note the recommended sampling parameters for reasoning mode: <code>temperature=0.6</code>, <code>top_p=0.95</code>.
    </div>

---
### Part 9: Benchmarking Performance

Before deploying a VLM in production, you need to know what it costs. Four knobs drive VLM performance:

- **ISL (input sequence length)**: mostly vision tokens - resolution × frames
- **OSL (output sequence length)**: how many tokens you ask the model to generate
- **Sampled frames**: how much of the video the model actually sees
- **Concurrency**: how many requests run at once

Let's measure each one.

#### 9.1 Latency: Time to First Token and Output Length

Two latency numbers matter: **TTFT** (time to first token - dominated by input processing) and **total time** (dominated by output length). We can measure both with a streaming request.

In [ ]:
# Measure TTFT and total latency with streaming
start_t = time.time()
first_token_t = None
n_chunks = 0

stream = vlm_client.chat.completions.create(
    model=vlm_model,
    messages=[
        {"role": "user",
         "content": [
            {"type": "video_url", "video_url": {"url": video_data_url}},
            {"type": "text", "text": "Describe this video in detail."},
         ]},
    ],
    max_tokens=512,
    temperature=0.4,
    top_p=0.4,
    stream=True,
)

for chunk in stream:
    if first_token_t is None:
        first_token_t = time.time()
    n_chunks += 1

total_time = time.time() - start_t
assert first_token_t is not None, "No tokens received - check the model server."
ttft = first_token_t - start_t

print(f"Time to First Token (TTFT): {ttft:.2f}s")
print(f"Total Time: {total_time:.2f}s")
print(f"Generation Time: {total_time - ttft:.2f}s for ~{n_chunks} tokens (~{n_chunks / max(total_time - ttft, 1e-6):.1f} tok/s)")

Now let's isolate the effect of **output length** by asking for the same video at three different `max_tokens` budgets. Input is identical each time - only the output length changes.

In [ ]:
# OSL sweep - same input, different output budgets
osl_results = []
for osl in [1, 128, 512]:
    start_t = time.time()
    vlm_client.chat.completions.create(
        model=vlm_model,
        messages=[
            {"role": "user",
             "content": [
                {"type": "video_url", "video_url": {"url": video_data_url}},
                {"type": "text", "text": "Describe this video." if osl > 1 else "Is anything moving in this video? Answer 'yes' or 'no'."},
             ]},
        ],
        max_tokens=osl,
        temperature=0.4,
        top_p=0.4,
    )
    elapsed = time.time() - start_t
    osl_results.append((osl, elapsed))
    print(f"max_tokens={osl:4d}  ->  {elapsed:.2f}s")

print("\nOutput length drives total latency - a 1-token verdict costs a fraction of a full caption.")

---
#### 9.2 Input Size: Sampled Frames

The other side of the equation is input size. For video, the biggest input knob is **how many frames the model samples**. The NIM defaults to 4 FPS; we can override it per request with `media_io_kwargs`.

In [ ]:
# Frame sampling sweep - fewer vs more frames from the same video
for fps in [1, 4, 8]:
    start_t = time.time()
    response = vlm_client.chat.completions.create(
        model=vlm_model,
        messages=[
            {"role": "user",
             "content": [
                {"type": "video_url", "video_url": {"url": video_data_url}},
                {"type": "text", "text": "Describe this video in one paragraph."},
             ]},
        ],
        max_tokens=256,
        temperature=0.4,
        top_p=0.4,
        extra_body={"media_io_kwargs": {"video": {"fps": fps}}},
    )
    elapsed = time.time() - start_t
    print(f"fps={fps}  ->  {elapsed:.2f}s")

print("\nMore sampled frames = more vision tokens = higher latency (but better temporal coverage).")

---
#### 9.3 Concurrency and Throughput

Finally, throughput: a serving engine batches concurrent requests, so total throughput grows with concurrency even as per-request latency degrades. Let's measure both by sending the same image request at increasing concurrency levels.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def one_request():
    start = time.time()
    vlm_client.chat.completions.create(
        model=vlm_model,
        messages=[
            {"role": "user",
             "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
                {"type": "text", "text": "Describe this image in one paragraph."},
             ]},
        ],
        max_tokens=128,
        temperature=0.4,
        top_p=0.4,
    )
    return time.time() - start

print(f"{'Concurrency':>12} | {'Wall Time':>9} | {'Avg Latency':>11} | {'Throughput':>10}")
print("-" * 55)
for n in [1, 2, 4]:
    start_t = time.time()
    with ThreadPoolExecutor(max_workers=n) as pool:
        latencies = list(pool.map(lambda _: one_request(), range(n)))
    wall = time.time() - start_t
    print(f"{n:>12} | {wall:>8.2f}s | {sum(latencies)/len(latencies):>10.2f}s | {n/wall:>7.2f} req/s")

Look at how the numbers move: per-request latency climbs with concurrency, but total throughput (requests per second) climbs too - the serving engine is batching the work.

This is exactly how production sizing works. For example, the published [VSS RT-VLM performance tables](https://docs.nvidia.com/vss/3.2.0/performance-rt-vlm.html) show one H100 sustaining roughly **51 concurrent streams for single-token alerting** but only **33 for full captioning** (measured at 10-second chunks, 80 frames at 448×448, with the FP8 checkpoint). Same model, same GPU - the difference is nothing but the OSL and concurrency trade-offs you just measured.

---
### Part 10: Tear Down

In this lab environment, cleanup is handled for you. When running on your own machine, stop and remove the NIM container to free the GPU:

```bash
docker stop cosmos3-reasoner && docker rm cosmos3-reasoner
```

The model cache in `~/.cache/nim` is preserved, so the next deployment starts much faster.

---
### Review

In this notebook you covered:
- What VLMs are and the **Cosmos 3** family (Edge / Nano / Super) - and how the Reasoner tower relates to the full model
- How the **Cosmos 3 Reasoner NIM** is deployed with a single `docker run`, and torn down when finished
- Image and video inference through the NIM's OpenAI-compatible API
- The vision token trade-off: more tokens means better quality, but slower inference and higher memory usage
- **Prompting**: role + focus + format + honesty beats a vague question every time
- **Structured outputs**: formatted reports, JSON events with timestamps, and single-token true/false and multiple-choice answers
- **Reasoning vs non-reasoning**: better grounding on complex judgments, at the cost of more generated tokens
- **Benchmarking**: TTFT vs total latency, and how OSL, sampled frames, and concurrency drive performance

In the next notebook, we'll move from *understanding* video to *generating* it - using Cosmos for video generation and Qwen Image Edit to augment and build datasets, which we'll then use to fine-tune a VLM in the final notebook.